# Sentiment Generation — Strategy & Judge Analysis

Compares all 4 GET generation strategies (forward/inverse × seeded/seedless) across:
- Model performance degradation (macro F1 vs real baseline)
- Sample yield and stability
- Judge filtering impact
- Model sensitivity (BERTweet vs Multilingual)

> **With vs Without Judge**: All current runs used LLM-as-judge. A without-judge section is prepared — add runs with `judge: enabled: false` to see the comparison.


In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
# Walk up from CWD until we find the project root (has framework/data/)
def _find_root():
    for p in [Path().resolve()] + list(Path().resolve().parents):
        if (p / 'framework' / 'data').exists():
            return p
    raise FileNotFoundError('Cannot find project root (no framework/data/ found)')

PROJECT_ROOT = _find_root()
RUNS_DIR     = PROJECT_ROOT / 'framework' / 'data' / 'runs' / 'sentiment'
NB_DIR       = Path().resolve()

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'RUNS_DIR     = {RUNS_DIR}  (exists: {RUNS_DIR.exists()})')

# ── Style ────────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})

STRATEGY_COLORS = {
    'forward_seeded':    '#2563eb',
    'forward_seedless':  '#7c3aed',
    'inverse_seeded':    '#059669',
    'inverse_seedless':  '#d97706',
}
MODEL_COLORS = {
    'bertweet':       '#2563eb',
    'multilingual':   '#dc2626',
}

SHORT_MODEL = {
    'finiteautomata/bertweet-base-sentiment-analysis': 'bertweet',
    'lxyuan/distilbert-base-multilingual-cased-sentiments-student': 'multilingual',
}

print('Notebook ready.')

In [ ]:
# ── 1. Load all runs ─────────────────────────────────────────────────────────
def strategy_label(meta):
    mode = meta.get('mode', '?')
    seed = 'seedless' if meta.get('seedless') else 'seeded'
    return f'{mode}_{seed}'

def has_judge(meta):
    j = meta.get('judge')
    return bool(j and j.get('model'))

def safe_mean(vals):
    vals = [v for v in vals if v is not None]
    return np.mean(vals) if vals else None

def safe_std(vals):
    vals = [v for v in vals if v is not None]
    return np.std(vals, ddof=1) if len(vals) > 1 else 0.0

# Collect run dirs from both new-style (RUNS_DIR/<run_id>/) and
# old-style (RUNS_DIR.parent/sentiment_corruption_*/<timestamp>/) layouts.
all_run_dirs = []

# New-style
if RUNS_DIR.exists():
    for sub in sorted(RUNS_DIR.iterdir()):
        if (sub / 'results.json').exists():
            all_run_dirs.append((sub, sub.name, False))

# Old-style
OLD_RUNS_BASE = RUNS_DIR.parent
for task_dir in sorted(OLD_RUNS_BASE.iterdir()):
    if not task_dir.is_dir() or not task_dir.name.startswith('sentiment_corruption'):
        continue
    for sub in sorted(task_dir.iterdir()):
        if (sub / 'results.json').exists():
            all_run_dirs.append((sub, f'{task_dir.name}/{sub.name}', True))

runs_raw = []
for d, run_id, is_old in all_run_dirs:
    r = json.loads((d / 'results.json').read_text(encoding='utf-8'))
    meta = r.get('meta', {})
    results = r.get('results', {})

    row_base = {
        'run_id':          run_id,
        'strategy':        strategy_label(meta),
        'mode':            meta.get('mode'),
        'seedless':        meta.get('seedless', False),
        'judge':           has_judge(meta),
        'gen_model':       meta.get('model', '?'),
        'judge_model':     (meta.get('judge') or {}).get('model', 'none'),
        'num_runs':        meta.get('runs_completed', 0),
        'samples_per_run': meta.get('effective_samples_per_run', []),
        'created':         meta.get('created', ''),
    }

    for model_full, scores in results.items():
        mshort = SHORT_MODEL.get(model_full, model_full.split('/')[-1][:20])
        gen = scores.get('generated') or {}
        real = scores.get('real') or {}
        runs_list = scores.get('runs', [])

        per_run_f1 = [rr.get('macro_f1') for rr in runs_list if rr.get('macro_f1') is not None]
        gen_f1_mean = gen.get('macro_f1')
        if isinstance(gen_f1_mean, dict):
            gen_f1_mean = gen_f1_mean.get('mean')

        real_f1 = real.get('macro_f1')
        if isinstance(real_f1, dict):
            real_f1 = real_f1.get('mean')

        row = {
            **row_base,
            'model':         mshort,
            'gen_f1_mean':   gen_f1_mean,
            'gen_f1_std':    safe_std(per_run_f1) if per_run_f1 else (gen.get('std') if isinstance(gen.get('std'), float) else None),
            'real_f1':       real_f1,
            'per_run_f1':    per_run_f1,
            'gen_acc':       gen.get('accuracy') if isinstance(gen.get('accuracy'), float) else (gen.get('accuracy') or {}).get('mean'),
            'real_acc':      real.get('accuracy') if isinstance(real.get('accuracy'), float) else None,
        }
        runs_raw.append(row)

df = pd.DataFrame(runs_raw)
df['f1_drop'] = df['real_f1'] - df['gen_f1_mean']
df['mean_samples'] = df['samples_per_run'].apply(lambda x: np.mean(x) if x else 0)
df['samples_requested'] = 150
df['yield_rate'] = df['mean_samples'] / df['samples_requested']
df['created'] = pd.to_datetime(df['created'], errors='coerce')

print(f'Loaded {len(df)} model×run combinations across {df["run_id"].nunique()} run folders.')
print(f'  With judge:    {df[df["judge"]==True]["run_id"].nunique()} unique runs')
print(f'  Without judge: {df[df["judge"]==False]["run_id"].nunique()} unique runs')
df[['run_id','strategy','judge','gen_model','num_runs','mean_samples','model','gen_f1_mean','real_f1']].to_string(index=False)

## 1. Run Overview

In [ ]:
# Pick best run per (strategy, judge) — latest creation date if duplicates
df_best = (
    df.sort_values('created', ascending=False)
      .drop_duplicates(subset=['strategy', 'judge', 'model'])
      .copy()
)

overview = (
    df_best[df_best['model'] == 'bertweet']
    [['run_id','strategy','judge','gen_model','num_runs','mean_samples','yield_rate']]
    .sort_values('strategy')
    .reset_index(drop=True)
)
overview.columns = ['Run ID','Strategy','Judge','Gen model','Completed runs','Eval samples/run','Yield']
overview['Eval samples/run'] = overview['Eval samples/run'].map(lambda x: f'{x:.0f}')
overview.style \
    .set_caption('Run overview — one row per strategy (best/latest run)') \
    .format({'Yield': '{:.0%}'}) \
    .background_gradient(subset=['Yield'], cmap='RdYlGn')

## 2. Macro F1 — Strategy Comparison

Lower synthetic F1 = the synthetic data is harder for the classifier → more effective adversarial data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
metrics = ['gen_f1_mean', 'f1_drop']
titles  = ['Synthetic Macro F1  (↓ harder for classifier)', 'F1 Drop vs Real Baseline  (↑ more degradation)']

for ax, metric, title in zip(axes, metrics, titles):
    for model, mdf in df_best.groupby('model'):
        mdf = mdf.sort_values('strategy')
        x = np.arange(len(mdf))
        offset = -0.2 if model == 'bertweet' else 0.2
        bars = ax.bar(x + offset, mdf[metric], width=0.35,
                      label=model, color=MODEL_COLORS[model], alpha=0.85)
        if metric == 'gen_f1_mean':
            # error bars from std
            stds = mdf['gen_f1_std'].fillna(0)
            ax.errorbar(x + offset, mdf[metric], yerr=stds,
                        fmt='none', color='black', capsize=4, linewidth=1.2)

    # real baseline line (same for all strategies)
    if metric == 'gen_f1_mean':
        for model, mdf in df_best.groupby('model'):
            real_val = mdf['real_f1'].iloc[0]
            ax.axhline(real_val, color=MODEL_COLORS[model], linestyle='--',
                       linewidth=1.2, alpha=0.6)

    strats = df_best[df_best['model'] == 'bertweet'].sort_values('strategy')['strategy'].tolist()
    ax.set_xticks(np.arange(len(strats)))
    ax.set_xticklabels([s.replace('_', '\n') for s in strats], fontsize=9)
    ax.set_title(title, fontsize=10, pad=8)
    ax.legend(fontsize=9)
    if metric == 'f1_drop':
        ax.set_ylabel('Δ F1 (real − synthetic)')
    else:
        ax.set_ylabel('Macro F1')

fig.suptitle('Strategy Comparison — Macro F1', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(NB_DIR / 'judge_analysis_f1_comparison.png', bbox_inches='tight')
plt.show()

## 3. Performance Degradation Heatmap

In [ ]:
pivot = df_best.pivot_table(index='strategy', columns='model', values='f1_drop', aggfunc='first')

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn_r',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'F1 drop (real − synthetic)'},
    vmin=0, vmax=0.5
)
ax.set_title('F1 Drop per Strategy × Model (higher = more degradation)', pad=10)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(NB_DIR / 'judge_analysis_heatmap.png', bbox_inches='tight')
plt.show()

print('\nRanking by mean F1 drop across models:')
rank = pivot.mean(axis=1).sort_values(ascending=False)
for strat, drop in rank.items():
    print(f'  {strat:<25} Δ={drop:.3f}')

## 4. Sample Yield — Judge Filtering Impact

Shows how many of the 150 requested samples survived to evaluation. Lower yield = judge filtered more = higher quality bar, but less data.

In [ ]:
# Per strategy, per run: effective eval samples
yield_rows = []
for _, row in df[df['model'] == 'bertweet'].iterrows():
    for run_idx, n in enumerate(row['samples_per_run']):
        yield_rows.append({
            'strategy': row['strategy'],
            'run_id':   row['run_id'],
            'judge':    row['judge'],
            'run':      run_idx + 1,
            'samples':  n,
        })
ydf = pd.DataFrame(yield_rows)

# Use best run per strategy
ydf_best = ydf[ydf['run_id'].isin(df_best['run_id'].unique())]

fig, ax = plt.subplots(figsize=(10, 4))
strats = sorted(ydf_best['strategy'].unique())
colors_run = ['#93c5fd', '#3b82f6', '#1d4ed8']

for i, strat in enumerate(strats):
    sub = ydf_best[ydf_best['strategy'] == strat].sort_values('run')
    for j, (_, r) in enumerate(sub.iterrows()):
        color = colors_run[j % len(colors_run)]
        ax.bar(i + (j - 1) * 0.25, r['samples'], width=0.22,
               color=color, alpha=0.9, label=f'Run {r["run"]}' if i == 0 else '')

ax.axhline(150, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='Requested (150)')
ax.set_xticks(range(len(strats)))
ax.set_xticklabels([s.replace('_', '\n') for s in strats])
ax.set_ylabel('Evaluable samples per run')
ax.set_title('Sample Yield per Strategy & Run  (red dashed = 150 requested)', pad=8)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(NB_DIR / 'judge_analysis_yield.png', bbox_inches='tight')
plt.show()

print('\nMean evaluable samples per run:')
for strat in strats:
    sub = ydf_best[ydf_best['strategy'] == strat]['samples']
    print(f'  {strat:<25} {sub.mean():.0f} ± {sub.std():.0f}  (yield {sub.mean()/150:.0%})')

## 5. Run Stability — Variance Analysis

High std = results are noisy across runs, need more runs for reliable estimates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, model in zip(axes, ['bertweet', 'multilingual']):
    sub = df_best[df_best['model'] == model].sort_values('strategy')
    strats = sub['strategy'].tolist()
    x = np.arange(len(strats))

    for xi, row in zip(x, sub.itertuples()):
        per_run = getattr(row, 'per_run_f1', []) or []
        if per_run:
            color = STRATEGY_COLORS.get(row.strategy, '#6b7280')
            ax.bar(xi, row.gen_f1_mean, color=color, alpha=0.75, width=0.5)
            ax.errorbar(xi, row.gen_f1_mean, yerr=row.gen_f1_std or 0,
                        fmt='none', color='black', capsize=6, linewidth=1.5)
            # individual run dots
            jitter = np.linspace(-0.12, 0.12, len(per_run))
            ax.scatter([xi + j for j in jitter], per_run,
                       color='black', s=30, zorder=5, alpha=0.7)

    ax.axhline(sub['real_f1'].iloc[0], color='red', linestyle='--',
               linewidth=1.2, alpha=0.7, label='Real baseline')
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace('_', '\n') for s in strats], fontsize=9)
    ax.set_title(f'{model} — F1 with ±1 std  (dots = individual runs)', fontsize=10)
    ax.set_ylabel('Macro F1')
    ax.legend(fontsize=9)

fig.suptitle('Run Stability — Mean ± Std per Strategy', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(NB_DIR / 'judge_analysis_stability.png', bbox_inches='tight')
plt.show()

print('\nStd summary (lower = more stable):')
for _, row in df_best[df_best['model'] == 'bertweet'].sort_values('gen_f1_std', ascending=False).iterrows():
    flag = ' ⚠ HIGH' if (row['gen_f1_std'] or 0) > 0.10 else ''
    print(f'  {row["strategy"]:<25} std={row["gen_f1_std"]:.4f}{flag}')

## 6. Model Sensitivity — BERTweet vs Multilingual

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for strat in sorted(df_best['strategy'].unique()):
    sub = df_best[df_best['strategy'] == strat]
    bt  = sub[sub['model'] == 'bertweet']['gen_f1_mean'].values
    ml  = sub[sub['model'] == 'multilingual']['gen_f1_mean'].values
    if len(bt) and len(ml):
        color = STRATEGY_COLORS.get(strat, '#6b7280')
        ax.scatter(bt[0], ml[0], s=160, color=color, zorder=5,
                   label=strat.replace('_', ' '))
        ax.annotate(strat.replace('_', '\n'), (bt[0], ml[0]),
                    textcoords='offset points', xytext=(6, 4), fontsize=8, color=color)

# Real baselines
bt_real = df_best[df_best['model'] == 'bertweet']['real_f1'].iloc[0]
ml_real = df_best[df_best['model'] == 'multilingual']['real_f1'].iloc[0]
ax.scatter(bt_real, ml_real, s=200, marker='*', color='black', zorder=6, label='Real baseline')
ax.annotate('Real', (bt_real, ml_real), textcoords='offset points', xytext=(6, 4), fontsize=9)

# Diagonal: equal degradation line
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]) - 0.02,
        max(ax.get_xlim()[1], ax.get_ylim()[1]) + 0.02]
ax.plot(lims, lims, '--', color='gray', alpha=0.4, linewidth=1, label='Equal degradation')

ax.set_xlabel('BERTweet Macro F1 (synthetic)')
ax.set_ylabel('Multilingual Macro F1 (synthetic)')
ax.set_title('Model Sensitivity — Which model degrades more?\n(bottom-left = harder for both)', pad=8)
ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(NB_DIR / 'judge_analysis_model_sensitivity.png', bbox_inches='tight')
plt.show()

## 7. With vs Without Judge

**Status**: all current runs were generated with LLM-as-judge enabled.  
To activate this section, run each strategy once with `judge: enabled: false` in the config.

In [ ]:
judged   = df[df['judge'] == True]
unjudged = df[df['judge'] == False]

if unjudged.empty:
    print('⚠  No without-judge runs found in', RUNS_DIR)
    print()
    print('To run without judge, set in config.yaml:')
    print('  judge:')
    print('    enabled: false')
    print()
    print('Then re-run each strategy once and this section will auto-populate.')
    print()
    print('Expected effect: without judge, F1 should be LOWER (more noise in data)')
    print('Judge improves data QUALITY but reduces QUANTITY (yield drops).')
    print()
    # Show judge drop rates as proxy: samples_per_run vs 150 requested
    print('Proxy — judge filtering impact (effective eval samples vs 150 requested):')
    judge_proxy = (
        df_best[df_best['model'] == 'bertweet']
        [['strategy','mean_samples','yield_rate','gen_f1_std']]
        .sort_values('yield_rate')
    )
    for _, r in judge_proxy.iterrows():
        print(f'  {r["strategy"]:<25} yield={r["yield_rate"]:.0%}  std={r["gen_f1_std"]:.3f}')
else:
    # Full with/without judge comparison
    fig, ax = plt.subplots(figsize=(11, 5))
    strats = sorted(df['strategy'].unique())
    x = np.arange(len(strats))
    width = 0.35

    for model in ['bertweet']:
        j_vals  = [judged[(judged['strategy']==s) & (judged['model']==model)]['gen_f1_mean'].mean() for s in strats]
        uj_vals = [unjudged[(unjudged['strategy']==s) & (unjudged['model']==model)]['gen_f1_mean'].mean() for s in strats]
        ax.bar(x - width/2, j_vals,  width, label='With judge',    color='#2563eb', alpha=0.85)
        ax.bar(x + width/2, uj_vals, width, label='Without judge', color='#94a3b8', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([s.replace('_', '\n') for s in strats])
    ax.set_ylabel('BERTweet Macro F1 (synthetic)')
    ax.set_title('With vs Without Judge — Effect on Data Quality')
    ax.legend()
    plt.tight_layout()
    plt.savefig(NB_DIR / 'judge_analysis_with_without.png', bbox_inches='tight')
    plt.show()

## 8. Forward Seedless — Label Coverage Issue

Seedless forward generation uses free-text error type names (e.g. "Negation", "None") instead of canonical keys (e.g. `negation_insertion`). Only canonical names map to ground-truth labels for evaluation.

In [ ]:
# Load generated samples to inspect error types
CANONICAL = {'negation_insertion', 'sentiment_flip_negative', 'sentiment_flip_positive',
             'sarcasm_injection', 'intensity_reduction', 'paraphrase'}

label_coverage = []
for d, run_id, _ in all_run_dirs:
    gen_dir = d / 'generated'
    if not gen_dir.exists():
        continue
    samples = []
    for jf in gen_dir.glob('*.json'):
        try:
            data = json.loads(jf.read_text(encoding='utf-8'))
            if isinstance(data, list):
                samples.extend(data)
        except Exception:
            pass
    if not samples:
        continue
    total = len(samples)
    canonical_count = sum(1 for s in samples if s.get('error_type','').lower() in CANONICAL)
    matching = df[df['run_id'] == run_id]
    strategy = matching['strategy'].iloc[0] if not matching.empty else '?'
    judge = matching['judge'].iloc[0] if not matching.empty else '?'
    label_coverage.append({
        'run_id':         run_id,
        'strategy':       strategy,
        'judge':          judge,
        'total_samples':  total,
        'canonical_rate': canonical_count / total if total else 0,
    })

lcdf = pd.DataFrame(label_coverage)
print('Error type label coverage per run:')
print(lcdf[['run_id','strategy','judge','total_samples','canonical_rate']].to_string(index=False))
print()
print('⚠  Old forward_seedless runs (without judge) have low canonical coverage')
print('   because the LLM wrote free-text error types. The fix (storing canonical')
print('   keys from the sampled distribution) is applied in the new run.')

## 9. Per-Run F1 Progression

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, model in zip(axes, ['bertweet', 'multilingual']):
    sub = df_best[df_best['model'] == model].sort_values('strategy')
    for _, row in sub.iterrows():
        per_run = row['per_run_f1']
        if not per_run:
            continue
        color = STRATEGY_COLORS.get(row['strategy'], '#6b7280')
        x = range(1, len(per_run) + 1)
        ax.plot(x, per_run, marker='o', color=color, linewidth=1.8,
                label=row['strategy'].replace('_', ' '))
    
    real_val = sub['real_f1'].iloc[0]
    ax.axhline(real_val, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='Real baseline')
    ax.set_xlabel('Run index')
    ax.set_ylabel('Macro F1')
    ax.set_title(f'{model}', fontsize=10)
    ax.set_xticks([1, 2, 3])
    ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')

fig.suptitle('Per-Run F1 Progression (consistent = stable strategy)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(NB_DIR / 'judge_analysis_progression.png', bbox_inches='tight')
plt.show()

## 10. Key Findings

In [ ]:
print('=' * 65)
print('KEY FINDINGS — Sentiment Strategy & Judge Analysis')
print('=' * 65)

print('\n── Real Baseline ──────────────────────────────────────────')
for model in ['bertweet', 'multilingual']:
    real_f1 = df_best[df_best['model'] == model]['real_f1'].iloc[0]
    print(f'  {model:<20} F1 = {real_f1:.4f}')

print('\n── Strategy Ranking (by mean F1 drop, both models) ────────')
pivot_drop = df_best.pivot_table(index='strategy', columns='model', values='f1_drop', aggfunc='first')
pivot_drop['mean_drop'] = pivot_drop.mean(axis=1)
for strat, row in pivot_drop.sort_values('mean_drop', ascending=False).iterrows():
    flag = ''
    std_val = df_best[(df_best['strategy'] == strat) & (df_best['model'] == 'bertweet')]['gen_f1_std'].values
    if len(std_val) and std_val[0] > 0.10:
        flag = '  ⚠ HIGH VARIANCE'
    print(f'  {strat:<25}  Δ={row["mean_drop"]:.3f}{flag}')

print('\n── Sample Yield ────────────────────────────────────────────')
for _, row in df_best[df_best['model'] == 'bertweet'].sort_values('yield_rate').iterrows():
    print(f'  {row["strategy"]:<25}  {row["mean_samples"]:.0f}/150  ({row["yield_rate"]:.0%})')

print('\n── Stability (BERTweet F1 std across runs) ─────────────────')
for _, row in df_best[df_best['model'] == 'bertweet'].sort_values('gen_f1_std', ascending=False).iterrows():
    status = '✓ stable' if (row['gen_f1_std'] or 0) < 0.06 else '⚠ unstable'
    print(f'  {row["strategy"]:<25}  std={row["gen_f1_std"]:.4f}  {status}')

print('\n── Label Coverage Issue (Forward Seedless) ─────────────────')
print('  Gemini generates free-text error types → low eval coverage.')
print('  Fix: pin error_type from sampled key, ignore LLM output field.')

print('\n── Recommendations ─────────────────────────────────────────')
print('  1. Add without-judge runs to quantify judge quality uplift.')
print('  2. Fix forward seedless label coverage before drawing conclusions.')
print('  3. Forward seeded is the most stable & high-yield judged strategy.')
print('  4. Inverse seeded shows strongest degradation — best adversarial data.')
print('  5. Multilingual model is generally MORE degraded than BERTweet,')
print('     suggesting it is less robust to synthetic sentiment shifts.')